# OO 5: Repetition - tickets en events

Herhalingsoefening waarin meerdere OO-concepten samenkomen.

## Gebruikte concepten

- Properties met validatie
- Static methods
- Dunder method `__str__`
- Abstract class en abstract method
- Inheritance
- Overriding via concrete subclasses

## Stap 1: ticket-ID valideren

Een ticket-ID is geldig als:

- het exact 8 karakters lang is
- de eerste 3 karakters hoofdletters zijn
- de laatste 5 karakters cijfers zijn

Voorbeeld geldig: `VIP12345`.

In [ ]:
def is_valid_ticket_id(ticket_id):
    return (
        isinstance(ticket_id, str)
        and len(ticket_id) == 8
        and ticket_id[:3].isalpha()
        and ticket_id[:3].isupper()
        and ticket_id[3:].isdigit()
    )


print(is_valid_ticket_id("VIP12345"))
print(is_valid_ticket_id("vip12345"))
print(is_valid_ticket_id("VIP12"))

## Stap 2: `Ticket` class

Waarom properties?

De ticket-ID mag niet zomaar elke waarde zijn. Door een setter te gebruiken wordt validatie automatisch uitgevoerd bij het aanmaken en aanpassen van een ticket.

Waarom static method?

De validatieregel hoort logisch bij `Ticket`, maar heeft geen bestaand ticket-object nodig.

In [ ]:
class Ticket:
    def __init__(self, ticket_id, ticket_type, price):
        self.ticket_id = ticket_id
        self.ticket_type = ticket_type
        self.price = price

    @property
    def ticket_id(self):
        return self.__ticket_id

    @ticket_id.setter
    def ticket_id(self, value):
        if not Ticket.is_valid_ticket_id(value):
            raise ValueError("Invalid ticket id")
        self.__ticket_id = value

    @staticmethod
    def is_valid_ticket_id(ticket_id):
        return (
            isinstance(ticket_id, str)
            and len(ticket_id) == 8
            and ticket_id[:3].isalpha()
            and ticket_id[:3].isupper()
            and ticket_id[3:].isdigit()
        )

    def __str__(self):
        return f"{self.ticket_id} - {self.ticket_type}: {self.price:.2f} EUR"


ticket = Ticket("VIP12345", "VIP", 149.99)
print(ticket)

## Stap 3: abstracte `Event` class

`Event` is een blueprint voor concrete events.

Waarom abstract?

Alle events moeten tickets kunnen beheren, maar elk type event maakt zijn eigen samenvatting. Daarom is `generate_event_summary()` abstract.

In [ ]:
from abc import ABC, abstractmethod


class Event(ABC):
    def __init__(self, name, max_capacity):
        self.name = name
        self.max_capacity = max_capacity
        self.__tickets = {}

    @property
    def ticket_count(self):
        return len(self.__tickets)

    def add_ticket(self, ticket):
        if not isinstance(ticket, Ticket):
            raise TypeError("Expected a Ticket object")
        if self.ticket_count >= self.max_capacity:
            raise ValueError("Event is full")
        if ticket.ticket_id in self.__tickets:
            raise ValueError("Ticket already exists")
        self.__tickets[ticket.ticket_id] = ticket

    def remove_ticket(self, ticket_id):
        if ticket_id not in self.__tickets:
            raise ValueError("Ticket not found")
        return self.__tickets.pop(ticket_id)

    def get_tickets(self):
        return list(self.__tickets.values())

    @abstractmethod
    def generate_event_summary(self):
        pass

## Stap 4: `Concert` subclass

Een concert is een event met meerdere artiesten. Het erft ticketbeheer van `Event` en implementeert alleen de specifieke samenvatting.

In [ ]:
class Concert(Event):
    def __init__(self, name, max_capacity, artists):
        super().__init__(name, max_capacity)
        self.artists = artists

    def generate_event_summary(self):
        artist_lines = "\n".join(f"- {artist}" for artist in self.artists)
        return f"Name: {self.name}\nTotal tickets: {self.ticket_count}\nArtists:\n{artist_lines}"


concert = Concert("Pukkelpop", 3, ["Beyonce", "Shakira"])
concert.add_ticket(Ticket("VIP12345", "VIP", 149.99))
concert.add_ticket(Ticket("STD12346", "Standard", 89.99))

print(concert.generate_event_summary())

## Stap 5: `PrivateEvent` subclass

Een private event heeft altijd maximumcapaciteit 50 en maar een artiest.

In [ ]:
class PrivateEvent(Event):
    def __init__(self, name, artist):
        super().__init__(name, max_capacity=50)
        self.artist = artist

    def generate_event_summary(self):
        return f"Name: {self.name}\nTotal tickets: {self.ticket_count}\nArtist: {self.artist}"


private_event = PrivateEvent("The island", "Shakira")
private_event.add_ticket(Ticket("PRS12345", "Private", 250.00))

print(private_event.generate_event_summary())

## Stap 6: fouten bewust testen

Voor examenvoorbereiding is het nuttig om ook foute cases te testen. Zo zie je waar je validatie precies zit.

In [ ]:
try:
    invalid_ticket = Ticket("vip12345", "VIP", 149.99)
except ValueError as error:
    print("Ticket error:", error)

try:
    small_concert = Concert("Small show", 1, ["Artist A"])
    small_concert.add_ticket(Ticket("AAA12345", "Standard", 20))
    small_concert.add_ticket(Ticket("BBB12345", "Standard", 20))
except ValueError as error:
    print("Event error:", error)